<a href="https://colab.research.google.com/github/e23189uop/Statistical-Learning-e23189/blob/main/assignment%2304_e23189/assignment04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
import io
import warnings
import scipy.stats as ss
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

warnings.filterwarnings('ignore')

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

class PlottingMethods:
    """
    Modular class to handle granular chart generation using Plotly.
    Returns figures for flexible embedding.
    """
    @staticmethod
    def create_bar(x, y, title="Bar Chart", xlabel="X", ylabel="Y", barmode='group'):
        """Creates a Plotly Bar Chart."""
        fig = px.bar(x=x, y=y, title=title, labels={'x': xlabel, 'y': ylabel}, barmode=barmode)
        return fig

    @staticmethod
    def create_pie(names, values, title="Pie Chart"):
        """Creates a Plotly Pie Chart."""
        fig = px.pie(names=names, values=values, title=title)
        return fig

    @staticmethod
    def create_histogram(x, title="Histogram", xlabel="X"):
        """Creates a Plotly Histogram."""
        fig = px.histogram(x=x, title=title, labels={'x': xlabel})
        return fig

    @staticmethod
    def create_heatmap(z, x, y, title="Association Heatmap"):
        """Creates a Plotly Heatmap for Correlation/Association Matrices."""
        fig = px.imshow(z, x=x, y=y, text_auto=".2f", title=title, aspect="auto", color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
        return fig


class DataInspector:
    """
    End-to-end tool for CSV data ingestion, advanced cleaning, feature engineering,
    and high-level statistical visualization.
    """
    def __init__(self):
        self.df = None
        self.plotter = PlottingMethods()

    # ==========================================
    # 1. Data Ingestion & Sanitization
    # ==========================================
    def upload_data(self, filepath=None):
        """Handles local file uploads and initializes DataFrame."""
        if IN_COLAB and filepath is None:
            print("Please upload your CSV file:")
            uploaded = files.upload()
            filename = list(uploaded.keys())[0]
            self.df = pd.read_csv(io.BytesIO(uploaded[filename]))
            print(f"Successfully uploaded {filename}")
        else:
            path = filepath or input("Enter the path to your CSV file: ")
            self.df = pd.read_csv(path)

        self._sanitize_data()

    def _sanitize_data(self):
        """Automatically converts garbage strings to NaN and auto-corrects types."""
        garbage_strings = ['?', 'n/a', 'N/A', 'NULL', 'null', ' ', '', '-', 'NA']
        self.df.replace(garbage_strings, np.nan, inplace=True)

        for col in self.df.columns:
            temp_col = pd.to_numeric(self.df[col], errors='coerce')
            # If conversion doesn't result in an entirely null column, convert it
            if not temp_col.isna().all():
                self.df[col] = temp_col

    # ==========================================
    # 2. Structural Analysis & Cleaning
    # ==========================================
    def data_summary(self):
        """Displays row/col counts, preview, and data types."""
        if self.df is None: return "Upload data first."
        print(f"Dataset Shape: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
        print("\n--- Column Types ---")
        num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=np.number).columns.tolist()
        print(f"Numerical Columns ({len(num_cols)}): {num_cols}")
        print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")
        print("\n--- Data Preview (First 20 rows) ---")
        display(self.df.head(20))

    def handle_missing_values(self, strategy='mean', constant_value=None):
        """Imputes missing values with mean, median, mode, or a constant."""
        for col in self.df.columns:
            if self.df[col].isnull().sum() == 0:
                continue
            if self.df[col].dtype in [np.float64, np.int64]:
                if strategy == 'mean':
                    self.df[col].fillna(self.df[col].mean(), inplace=True)
                elif strategy == 'median':
                    self.df[col].fillna(self.df[col].median(), inplace=True)
                elif strategy == 'constant' and constant_value is not None:
                    self.df[col].fillna(constant_value, inplace=True)
            else:
                # Mode for categorical
                self.df[col].fillna(self.df[col].mode()[0], inplace=True)
        print(f"Missing values imputed using '{strategy}' strategy.")

    def remove_duplicates(self):
        """Prunes exact row matches."""
        initial_shape = self.df.shape[0]
        self.df.drop_duplicates(inplace=True)
        print(f"Removed {initial_shape - self.df.shape[0]} duplicate rows.")

    def handle_outliers(self, columns, action='flag'):
        """IQR-based outlier detection system to flag or drop outliers."""
        if isinstance(columns, str): columns = [columns]

        for col in columns:
            if self.df[col].dtype not in [np.float64, np.int64]: continue
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            condition = (self.df[col] < lower_bound) | (self.df[col] > upper_bound)
            if action == 'drop':
                self.df = self.df[~condition]
            elif action == 'flag':
                self.df[f'{col}_outlier_flag'] = np.where(condition, 1, 0)
        print(f"Outliers processed for columns {columns} with action '{action}'.")

    def delete_rows(self, indices_str):
        """Interactive targeted deletion for rows (comma separated)."""
        indices = [int(i.strip()) for i in indices_str.split(',') if i.strip().isdigit()]
        self.df.drop(index=indices, inplace=True, errors='ignore')

    def delete_columns(self, cols_str):
        """Interactive targeted deletion for columns (comma separated)."""
        cols = [c.strip() for c in cols_str.split(',')]
        self.df.drop(columns=cols, inplace=True, errors='ignore')

    # ==========================================
    # 3. Feature Engineering (Normalization)
    # ==========================================
    def extract_normalized_numeric_data(self, strategy='standard'):
        """Extracts and scales numeric data (minmax, standard, robust)."""
        num_df = self.df.select_dtypes(include=np.number)
        if strategy == 'minmax':
            scaler = MinMaxScaler()
        elif strategy == 'robust':
            scaler = RobustScaler()
        else:
            scaler = StandardScaler()

        scaled_data = scaler.fit_transform(num_df)
        return pd.DataFrame(scaled_data, columns=num_df.columns, index=self.df.index)

    def extract_normalized_categorical_data(self, strategy='onehot'):
        """Extracts and encodes categorical data (onehot, ordinal, uniform)."""
        cat_df = self.df.select_dtypes(exclude=np.number)
        if strategy == 'ordinal' or strategy == 'uniform':
            encoder = OrdinalEncoder()
            encoded = encoder.fit_transform(cat_df)
            encoded_df = pd.DataFrame(encoded, columns=cat_df.columns, index=self.df.index)
            if strategy == 'uniform':
                # Scale between 0 and 1
                encoded_df = encoded_df / encoded_df.max()
            return encoded_df
        else:
            encoder = OneHotEncoder(sparse_output=False, drop='first')
            encoded = encoder.fit_transform(cat_df)
            col_names = encoder.get_feature_names_out(cat_df.columns)
            return pd.DataFrame(encoded, columns=col_names, index=self.df.index)

    def merge_datasets(self, num_strategy='standard', cat_strategy='onehot'):
        """Merges scaled numerical and encoded categorical sets."""
        num_scaled = self.extract_normalized_numeric_data(strategy=num_strategy)
        cat_encoded = self.extract_normalized_categorical_data(strategy=cat_strategy)
        return pd.concat([num_scaled, cat_encoded], axis=1)

    # ==========================================
    # 4. Interactive Visualization
    # ==========================================
    def plot_univariate_numeric(self, column):
        """Plots a 3-panel subplot: Box, Scatter, and Histogram."""
        if self.df[column].dtype not in [np.float64, np.int64]: return

        fig = make_subplots(rows=1, cols=3, subplot_titles=("Box Plot", "Index Scatter", "Histogram"))
        fig.add_trace(go.Box(x=self.df[column], name=column), row=1, col=1)
        fig.add_trace(go.Scatter(x=self.df.index, y=self.df[column], mode='markers'), row=1, col=2)
        fig.add_trace(go.Histogram(x=self.df[column]), row=1, col=3)
        fig.update_layout(title_text=f"Univariate Analysis: {column}", showlegend=False)
        fig.show()

    def plot_relationship(self, col1, col2):
        """Smart relational plot detecting datatypes (Num-Num, Cat-Num, Cat-Cat)."""
        is_col1_num = self.df[col1].dtype in [np.float64, np.int64]
        is_col2_num = self.df[col2].dtype in [np.float64, np.int64]

        if is_col1_num and is_col2_num:
            # Num-Num: Scatter with OLS Trendline
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"Scatter: {col1} vs {col2}")
        elif not is_col1_num and not is_col2_num:
            # Cat-Cat: Grouped Bar Chart
            counts = self.df.groupby([col1, col2]).size().reset_index(name='count')
            fig = self.plotter.create_bar(x=counts[col1], y=counts['count'], title=f"Grouped Bar: {col1} vs {col2}", barmode='group')
        else:
            # Cat-Num: Box plot with points
            c_cat, c_num = (col1, col2) if not is_col1_num else (col2, col1)
            fig = px.box(self.df, x=c_cat, y=c_num, points="all", title=f"Boxplot: {c_cat} vs {c_num}")

        fig.show()

    def plot_categorical_frequency(self, column):
        """Displays frequency bar charts for categoricals with percentage labels."""
        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, 'count']
        counts['percentage'] = (counts['count'] / counts['count'].sum() * 100).round(2).astype(str) + '%'

        fig = px.bar(counts, x=column, y='count', text='percentage', title=f"Frequency of {column}")
        fig.show()

    # ==========================================
    # 5. Deep Statistical Insights
    # ==========================================
    def _cramers_v(self, x, y):
        """Calculates Cramér's V for Categorical-Categorical correlation."""
        confusion_matrix = pd.crosstab(x, y)
        chi2 = ss.chi2_contingency(confusion_matrix)[0]
        n = confusion_matrix.sum().sum()
        phi2 = chi2 / n
        r, k = confusion_matrix.shape
        phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
        rcorr = r - ((r-1)**2)/(n-1)
        kcorr = k - ((k-1)**2)/(n-1)
        if min((kcorr-1), (rcorr-1)) == 0: return 0.0
        return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

    def _correlation_ratio(self, categories, measurements):
        """Calculates Eta (Correlation Ratio) for Categorical-Numerical correlation."""
        fcat, _ = pd.factorize(categories)
        cat_num = np.max(fcat)+1
        y_avg_array = np.zeros(cat_num)
        n_array = np.zeros(cat_num)

        valid_idx = ~pd.isna(categories) & ~pd.isna(measurements)
        categories, measurements = categories[valid_idx], measurements[valid_idx]

        for i in range(0, cat_num):
            cat_measures = measurements[fcat == i]
            n_array[i] = len(cat_measures)
            y_avg_array[i] = np.average(cat_measures) if len(cat_measures) > 0 else 0

        y_total_avg = np.sum(np.multiply(y_avg_array, n_array)) / np.sum(n_array)
        numerator = np.sum(np.multiply(n_array, np.power(np.subtract(y_avg_array, y_total_avg), 2)))
        denominator = np.sum(np.power(np.subtract(measurements, y_total_avg), 2))
        return 0.0 if denominator == 0 else np.sqrt(numerator / denominator)

    def plot_all_associations_heatmap(self):
        """Unified Heatmap mapping Pearson, Cramer's V, and Eta appropriately."""
        cols = self.df.columns
        n = len(cols)
        corr_matrix = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)

        for i, col1 in enumerate(cols):
            for j, col2 in enumerate(cols):
                is_num1 = self.df[col1].dtype in [np.float64, np.int64]
                is_num2 = self.df[col2].dtype in [np.float64, np.int64]

                if is_num1 and is_num2:
                    corr_matrix.loc[col1, col2] = self.df[col1].corr(self.df[col2])
                elif not is_num1 and not is_num2:
                    corr_matrix.loc[col1, col2] = self._cramers_v(self.df[col1], self.df[col2])
                else:
                    c_cat, c_num = (col1, col2) if not is_num1 else (col2, col1)
                    corr_matrix.loc[col1, col2] = self._correlation_ratio(self.df[c_cat], self.df[c_num])

        fig = self.plotter.create_heatmap(corr_matrix.values, list(cols), list(cols), "Unified Associations Heatmap")
        fig.show()

In [17]:
# 1. Initialize the instance
inspector = DataInspector()

# 2. Upload Data
inspector.upload_data()

# 3. Clean and Summary
inspector.data_summary()
inspector.delete_columns('Name')

# 4. Impute missing values
inspector.handle_missing_values(strategy='mean')

# 5. Outlier Detection
inspector.handle_outliers(['Age', 'Fare'], action='flag')

# 6. Feature Engineering
final_df = inspector.merge_datasets(num_strategy='robust', cat_strategy='onehot')
display(final_df.head())

# 7. Visualizations (Update this for your specific dataset)
inspector.plot_univariate_numeric('Age')
inspector.plot_relationship('Survived', 'Fare')
inspector.plot_categorical_frequency('Sex')

# 8. Deep Statistical Insights
inspector.plot_all_associations_heatmap()

Please upload your CSV file:


Saving titanic.csv to titanic (4).csv
Successfully uploaded titanic (4).csv
Dataset Shape: 887 rows, 8 columns

--- Column Types ---
Numerical Columns (6): ['Survived', 'Pclass', 'Age', 'Siblings/Spouses Aboard', 'Parents/Children Aboard', 'Fare']
Categorical Columns (2): ['Name', 'Sex']

--- Data Preview (First 20 rows) ---


,Survived,Pclass,Name,Sex,Age,Siblings/Spouses Aboard,Parents/Children Aboard,Fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500
5,0,3,Mr. James Moran,male,27.0,0,0,8.4583
6,0,1,Mr. Timothy J McCarthy,male,54.0,0,0,51.8625
7,0,3,Master. Gosta Leonard Palsson,male,2.0,3,1,21.0750
8,1,3,Mrs. Oscar W (Elisabeth Vilhelmina Berg) Johnson,female,27.0,0,2,11.1333
9,1,2,Mrs. Nicholas (Adele Achem) Nasser,female,14.0,1,0,30.0708


Missing values imputed using 'mean' strategy.
Outliers processed for columns ['Age', 'Fare'] with action 'flag'.


,Survived,Pclass,Age,Siblings/Spouses Aboard,Parents/Children Aboard,Fare,Age_outlier_flag,Fare_outlier_flag,Sex_male
0,0.0,0.0,-0.338028,1.0,0.0,-0.310359,0.0,0.0,1.0
1,1.0,-2.0,0.563380,1.0,0.0,2.448211,0.0,1.0,0.0
2,1.0,0.0,-0.112676,0.0,0.0,-0.281279,0.0,0.0,0.0
3,1.0,-2.0,0.394366,1.0,0.0,1.664870,0.0,0.0,0.0
4,0.0,0.0,0.394366,0.0,0.0,-0.275894,0.0,0.0,1.0
